In [11]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder.master("local[*]").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/11 17:36:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read\
	.option("header", "true")\
	.parquet('fhvhv_tripdata_2021-01.parquet')

In [4]:
df = df.repartition(24)

In [6]:
df.write.parquet("fhvhv/2021/01", mode="overwrite")

26/04/11 17:38:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/11 17:38:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/11 17:39:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/11 17:39:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [7]:
df = spark.read.parquet("fhvhv/2021/01")

In [8]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

In [9]:
df.select('PULocationID', 'DOLocationID').filter(df.hvfhs_license_num == 'HV0003').show(5)

+------------+------------+
|PULocationID|DOLocationID|
+------------+------------+
|          22|         261|
|         189|         181|
|          56|          75|
|         169|          69|
|         169|          69|
+------------+------------+
only showing top 5 rows


In [13]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)).select('pickup_datetime', 'pickup_date').show(10)

+-------------------+-----------+
|    pickup_datetime|pickup_date|
+-------------------+-----------+
|2021-01-05 12:42:09| 2021-01-05|
|2021-01-02 14:16:06| 2021-01-02|
|2021-01-06 08:26:45| 2021-01-06|
|2021-01-28 01:07:33| 2021-01-28|
|2021-01-17 09:00:11| 2021-01-17|
|2021-01-29 22:46:05| 2021-01-29|
|2021-01-14 22:58:21| 2021-01-14|
|2021-01-10 15:46:48| 2021-01-10|
|2021-01-21 05:45:15| 2021-01-21|
|2021-01-12 14:34:31| 2021-01-12|
+-------------------+-----------+
only showing top 10 rows


In [15]:
spark.read.option('header', 'true').csv('data/raw/yellow/2021/*/')

26/04/11 18:27:38 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/yellow/2021/*/.
java.io.FileNotFoundException: File data/raw/yellow/2021/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.

Py4JJavaError: An error occurred while calling o112.csv.
: org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:///Users/bastienwinant/Desktop/projects/data-engineering-zoomcamp/06-spark_batch_processing/data/raw/yellow/2021/08/yellow_tripdata_2021_08.csv.gz.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:911)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:141)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeSta

In [ ]:
spark.stop()